# Falhas silenciosas — o balde invisível

**Notebook genérico:** roda sobre qualquer extração apontada por `TRACE` em `base_pipeline.py`, como o notebook da
esteira. Só contagens, nomes de papel e de ferramenta e motivos mascarados; os casos (com `exec_id`) vão para
`resultados/evidencia/silenciosas/` (git-ignored) e não saem da máquina.

**Onde este notebook fica no desenho** (decisão de 01/10, `../../plano-atual.md` §1): um **funil de detecção único**
separa os steps em dois baldes, sem sobreposição, e cada balde é classificado à parte, com regras de entrada próprias e o
**mesmo catálogo de unidades**; no fim, a consolidação junta os dois.

| Balde | O que entra | Regras de entrada | Onde |
|---|---|---|---|
| **visível** | step com exceção (`ActionStep.error`) | classe + mensagem da exceção → família → mecanismo | `analise_trace_esteira_juridica.ipynb` |
| **invisível** | step **sem** exceção com `Error calling tool '<nome>'` no texto | grupo do motivo → forma do argumento → mecanismo | **este notebook** |
| consolidação | as tabelas de ocorrências dos dois | triagem final das unidades | `consolidacao_unidades.ipynb` (plano 4.2b) |

**Por que um balde próprio:** a ferramenta falha, o wrapper devolve a falha como **texto**, o Python segue e o step fica
com `error: null`. Para uma taxonomia construída sobre a exceção, esse step foi "ok". Nas duas bases, ~90% das falhas
de ferramenta são assim (ledger, Etapa 10c).

**Docs deste balde:** racionais [`../docs/13-racionais-falhas-silenciosas.md`](../docs/13-racionais-falhas-silenciosas.md) ·
relatório [`../docs/14-relatorio-falhas-silenciosas.md`](../docs/14-relatorio-falhas-silenciosas.md) ·
procedimento [`../docs/15-procedimento-falhas-silenciosas.md`](../docs/15-procedimento-falhas-silenciosas.md).
Seções §13.x (cada notebook é dono de um intervalo; README da pasta, roadmap item 26). As medidas
de §13.1–§13.6 são as mesmas funções do `drill_down.py silenciosas` (`base_pipeline.py`): o notebook e o comando
não divergem.

In [1]:
import json, re, ast, warnings
import pandas as pd, numpy as np
from collections import Counter, defaultdict
from base_pipeline import *   # espinha compartilhada desta pasta — ver base_pipeline.py

df = carregar_trace()
steps, RAW, execs = explodir_memoria(df)
E = classificar_erros(steps)
EU = montar_unidades(E)          # o balde visível — aqui só para cruzar ([2], [3]), não é recontado
F = falhas_silenciosas(df)       # uma linha por (step, ferramenta) chamada ou que falhou
sil = F[F["falha"] == "silenciosa"]
print(f"{BASE_ID}: {len(df)} execuções · {len(steps):,} ActionSteps · {len(E)} erros com exceção")

base1: 1000 execuções · 5,781 ActionSteps · 498 erros com exceção


## 13.1 · O funil: o que vai para cada balde

A regra de separação é por step e não deixa sobra nem sobreposição:

- step com exceção → **visível** (inclusive quando a mensagem é `Error calling tool`: são as falhas de ferramenta
  "com exceção", e ficam só lá);
- step sem exceção e com `Error calling tool '<nome>'` na observação ou no `action_output` → **invisível**;
- o resto → ok.

Limite declarado: só a forma `Error calling tool`. Validações que a ferramenta devolve com outro texto, e resultados
vazios ou errados sem mensagem, não entram.

In [2]:
passo = ["exec_id", "role", "idx"]
st_sil = sil[passo].drop_duplicates()
com_ferr = F[passo].drop_duplicates()
print("o funil, por step:")
print(f"  steps no total ........................... {len(steps):>7,}")
print(f"  → visível (com exceção) .................. {steps['err_type'].notna().sum():>7,}")
print(f"  → invisível (falha silenciosa) ........... {len(st_sil):>7,}")
print(f"  sobreposição visível × invisível ......... "
      f"{len(st_sil.merge(steps[steps['err_type'].notna()][passo], on=passo)):>7}  (tem de ser 0)")
print()
print(f"só os steps com ferramenta declarada (chamada ou falha): {len(com_ferr):,}")
n_exc, n_sil = (F["falha"] == "excecao").sum(), len(sil)
print(f"  falhas de ferramenta com exceção: {n_exc} · silenciosas: {n_sil} "
      f"→ {n_sil / max(n_exc + n_sil, 1):.0%} das falhas de ferramenta são invisíveis")
print(f"  silenciosas em {sil['exec_id'].nunique()} execuções, {sil['mes'].nunique()} meses, "
      f"{sil['role'].nunique()} papéis, {sil['ferramenta'].nunique()} ferramentas")

o funil, por step:
  steps no total ...........................   5,781
  → visível (com exceção) ..................     498
  → invisível (falha silenciosa) ...........     119
  sobreposição visível × invisível .........       0  (tem de ser 0)

só os steps com ferramenta declarada (chamada ou falha): 3,373
  falhas de ferramenta com exceção: 9 · silenciosas: 120 → 93% das falhas de ferramenta são invisíveis
  silenciosas em 90 execuções, 10 meses, 13 papéis, 22 ferramentas


## 13.2 · Por ferramenta

`% silenciosa` é sobre os steps em que a ferramenta foi chamada ou falhou. Uma ferramenta com poucas chamadas e
percentual alto é indício, não medida.

In [3]:
g = F.groupby("ferramenta")
tab = pd.DataFrame({"steps": g.size(),
                    "com exceção": g["falha"].apply(lambda s: (s == "excecao").sum()),
                    "silenciosas": g["falha"].apply(lambda s: (s == "silenciosa").sum())})
tab = tab[(tab["com exceção"] + tab["silenciosas"]) > 0].sort_values("silenciosas", ascending=False)
tab["% silenciosa"] = (tab["silenciosas"] / tab["steps"] * 100).round().astype(int)
tab["papéis"] = [", ".join(sorted(sil.loc[sil["ferramenta"] == t, "role"].unique())) or "—" for t in tab.index]
tab["meses"] = [sil.loc[sil["ferramenta"] == t, "mes"].nunique() for t in tab.index]
display(tab)

,steps,com exceção,silenciosas,% silenciosa,papéis,meses
ferramenta,,,,,,
get_available_documents,667,0,57,9,"ConversationAgent, managerAgent",9
compare_documents,48,1,13,27,ConversationAgent,4
answer_question_using_documents,744,0,11,1,ConversationAgent,3
table_statistical_summary,10,0,8,80,ConversationAgent,5
busca_obf,44,0,6,14,RoteadorCivel,2
trigger_worker_execution,78,0,4,5,WorkflowManager,3
get_reclamacao_text,28,0,3,11,RespostaBacen,2
calculo_correcoes_monetarias,16,1,2,12,CalculoCivel,1
get_docs_from_filters,16,0,2,12,OBFCivel,1


## 13.3 · O motivo: grupo e dono

`motivo_da_falha()` põe o texto depois de `Error calling tool '<nome>':` num de 6 grupos, por palavras-chave escritas
**olhando as duas bases** (cada regra anota a base de origem em `MOTIVO_REGRAS`). O que não casa vai para
`nao_reconhecido` e é lido como cobertura, como o resíduo do balde visível.

O **dono** é quem precisa mudar para a falha não acontecer. O rótulo é **genérico**: `json_invalido` fica "a conferir"
mesmo onde já foi conferido, porque em outra base o dono pode ser a ferramenta — quem decide em cada base é a §13.6.
`sem_resultado` e `fora_da_cobertura` não são falha real (a resposta certa é "não encontrei" / "fora do escopo").

In [4]:
fal = F[F["falha"].notna()]
grupos = pd.DataFrame([{"grupo": grp,
                        "silenciosas": (fal[fal["grupo"] == grp]["falha"] == "silenciosa").sum(),
                        "com exceção": (fal[fal["grupo"] == grp]["falha"] == "excecao").sum(),
                        "falha real": grp in GRUPOS_FALHA_REAL,
                        "ferramentas principais": ", ".join(f"{t} {n}" for t, n in
                                                            fal[fal["grupo"] == grp]["ferramenta"].value_counts().head(3).items()),
                        "dono": dono}
                       for grp, dono in DONO_DO_GRUPO.items()]).set_index("grupo")
display(grupos[(grupos["silenciosas"] + grupos["com exceção"]) > 0])

,silenciosas,com exceção,falha real,ferramentas principais,dono
grupo,,,,,
sem_resultado,43,0,False,"get_available_documents 38, get_reclamacao_tex...",ninguém — não é falha
fora_da_cobertura,2,6,False,"resposta_orgao_regulador 3, draft_resposta 2, ...",negócio — pedido fora do que a ferramenta cobre
argumento_do_agente,41,3,True,"get_available_documents 13, compare_documents ...",agente — candidato a memória que a taxonomia n...
plataforma,19,0,True,"answer_question_using_documents 4, trigger_wor...",plataforma / ferramenta
json_invalido,6,0,True,busca_obf 6,a conferir: agente ou ferramenta
nao_reconhecido,9,0,True,"get_available_documents 6, validador_calculo 1...",cobertura — sem regra de motivo


## 13.4 · O que vem depois de uma falha silenciosa

- **[2]** o 1º erro com exceção do mesmo papel em até 3 steps depois da falha. "(nenhum erro)" dominante = a falha
  passa sem deixar rastro no balde visível.
- **[3]** quanto dos erros de contrato de retorno (`U_tipo_retorno`, `U_contrato_dict`, `U_campo_inexistente`) vem logo
  depois de uma falha silenciosa no mesmo papel. Se fosse a maioria, essas memórias seriam efeito da plataforma, não
  lição para o agente. Não é (91–94% nas duas bases): continuam memória candidata.

In [5]:
print("[2] 1º erro do mesmo papel em até 3 steps depois de cada falha silenciosa:")
display(Counter(erro_depois_da_falha(sil, EU)).most_common(10))
print("[3] erros de contrato de retorno precedidos por falha silenciosa (até 3 steps antes, mesmo papel):")
display(pd.DataFrame([{"unidade": u, "erros": n, "precedidos": p, "%": round(p / n * 100) if n else 0}
                      for u, (n, p) in contrato_precedido(sil, EU).items()]).set_index("unidade"))

[2] 1º erro do mesmo papel em até 3 steps depois de cada falha silenciosa:


[('(nenhum erro)', 107),
 ('U_tipo_retorno', 4),
 ('H_bloco_code', 2),
 ('U_sandbox', 2),
 ('X_causa_nao_identificada', 1),
 ('U_contrato_dict', 1),
 ('U_texto_literal', 1),
 ('U_nome_inventado', 1),
 ('U_arg_nomeado', 1)]

[3] erros de contrato de retorno precedidos por falha silenciosa (até 3 steps antes, mesmo papel):


,erros,precedidos,%
unidade,,,
U_tipo_retorno,47,4,9
U_contrato_dict,96,1,1
U_campo_inexistente,10,0,0


## 13.5 · Candidato a sucesso falso

Só as falhas reais (`argumento_do_agente`, `plataforma`, `json_invalido`, `nao_reconhecido`). Candidato = o papel
entregou `final_answer` — no próprio step da falha ou depois, **na mesma chamada do papel** — sem nenhuma chamada sem
falha da mesma ferramenta no meio (Ajuste 11, 02/10: antes, a falha no próprio step final ficava fora e um final de
outra chamada entrava). **É teto:** a resposta pode não depender daquele dado — conferir no caso
(`drill_down.py caso <exec_id> <papel>`).

In [6]:
real = sil[sil["grupo"].isin(GRUPOS_FALHA_REAL)]
cand = sucesso_falso_candidato(F, sil)
sf = pd.DataFrame({"falhas reais": real["grupo"].value_counts(),
                   "candidatas a sucesso falso": real.loc[cand, "grupo"].value_counts()}).fillna(0).astype(int)
sf["%"] = (sf["candidatas a sucesso falso"] / sf["falhas reais"] * 100).round().astype(int)
print(f"{cand.sum()} de {len(real)} falhas reais ({len(real)} de {len(sil)} silenciosas)")
display(sf.sort_values("falhas reais", ascending=False))

54 de 75 falhas reais (75 de 120 silenciosas)


,falhas reais,candidatas a sucesso falso,%
grupo,,,
argumento_do_agente,41,30,73
plataforma,19,16,84
nao_reconhecido,9,8,89
json_invalido,6,0,0


## 13.6 · A forma do argumento — o dono do `json_invalido`, e o gesto do `repr_colado`

Para cada falha silenciosa, `forma_argumento()` lê **só o tipo** de como o 1º argumento foi passado no código do step:
`json.dumps(...)` ou a variável que outra ferramenta devolveu → o JSON quebrado viria da ferramenta; `str(...)`, dict
ou string montados à mão → o agente. "**Colado**" usa o critério do `repr_colado` (`submecanismo()`): ≥2 pares
`"chave": "texto"` e ≥2 dessas chaves já impressas numa observação anterior do mesmo papel — o agente cola o print de
um retorno em vez de passar a variável.

Nas duas bases, o `json_invalido` do `busca_obf` é inteiro esse gesto (base 1: 6/6; base 2: 86/86): é o `repr_colado`
no canal silencioso. Quando a colagem quebra a sintaxe, vira `SyntaxError` e cai no balde visível; quando é Python
válido dentro de `str()`, vira JSON inválido e cai aqui (ledger Etapa 10c, "Achado (01/10)").

In [7]:
sil = sil.assign(forma=formas_das_falhas(df, sil))
display(sil[sil["grupo"] == "json_invalido"].groupby(["ferramenta", "forma"]).size().to_frame("falhas"))
print("todas as falhas silenciosas, por grupo e forma:")
display(sil.groupby(["grupo", "forma"]).size().sort_values(ascending=False).to_frame("falhas"))

,,falhas
ferramenta,forma,
busca_obf,str(dict/lista colado de um retorno impresso) — gesto do repr_colado,6


todas as falhas silenciosas, por grupo e forma:


falhas
grupo               forma                                                     
sem_resultado       string literal montada pelo agente                      39
argumento_do_agente string literal montada pelo agente                      23
                    variável montada pelo agente                            12
plataforma          variável montada pelo agente                             8
                    string literal montada pelo agente                       8
nao_reconhecido     string literal montada pelo agente                       6
json_invalido       str(dict/lista colado de um retorno impresso) —...       6
argumento_do_agente string literal colada de um retorno impresso — ...       4
plataforma          json.dumps(...)                                          2
sem_resultado       sem chamada visível no código                            2
                    variável montada pelo agente                             2
nao_reconhecido     variável montada pelo agente                             2
fora_da_cobertura   variável montada pelo agente                             2
plataforma          string literal colada de um retorno impresso — ...       1
nao_reconhecido     string literal colada de um retorno impresso — ...       1
argumento_do_agente variável devolvida por str(...)                          1
                    outro                                                    1

## 13.7 · Detectores de comportamento (o ponto cego além do `Error calling tool`)

Classificando as categorias do TRAIL por visibilidade a exceção — mapeamento nosso, não uma medida que o paper
publica — **≥59% caem em tipos que esta metodologia não pegaria**. Os detectores
abaixo são determinísticos (AST), nenhum levanta exceção, e abrem parte desse ponto cego.

**Como ler cada detector** (testes de robustez em [`../docs/03-procedimento-validacao.md`](../docs/03-procedimento-validacao.md) §1.6):

1. **Result-Ignore** — retorno de ferramenta atribuído a uma variável que nunca é lida depois.
2. **RAC** — chamada idêntica repetida na trajetória, em steps sem erro. Número conservador: usa o inventário
   declarado no system prompt e exclui `final_answer`; muda bastante conforme essas duas escolhas.
3. **Reasoning-action mismatch — retirado, não usar.** O *thought* é escrito antes do código executar, então não
   há contradição a detectar; o número só aparece para registro.
4. **Tool-Skip — provisório.** Sensível ao inventário de ferramentas: é indício, não medida; inspecionar os casos
   um a um com `drill_down.py`.
5. **Ferramentas chamadas × declaradas** — a diferença são funções auxiliares que o próprio agente define.

*Migrado do §7 do notebook da esteira em 01/10 (plano 4.2), com o mesmo código: os números têm de bater com os que
aquele notebook publicava (base 1: inventário 90; Result-Ignore 103/3.053; RAC 125; Tool-Skip 10/840). Estes detectores
não dependem do texto `Error calling tool`: medem comportamento — retorno ignorado, chamada repetida, ferramenta não
chamada.*

In [8]:
import os, sys
sys.path.insert(0, os.path.join("..", ".."))   # analysis/base_utils.py — primitivas genéricas (roadmap item 26)
from base_utils import agrupar_por_execucao, inventario_ferramentas
warnings.filterwarnings("ignore", category=SyntaxWarning)   # ast.parse do código do agente; não muda nenhum número
bykey = agrupar_por_execucao(RAW)   # (exec_id, role) -> steps em ordem de idx; antes vinha da §6 da esteira

# INVENTÁRIO AUTORITATIVO: as ferramentas que o system prompt de cada papel DECLARA,
# como assinaturas `def nome(...)`. Extrair daqui, e não da observação das chamadas —
# um teste de robustez mostrou que a lista observada confundia ferramenta declarada com
# função auxiliar que o próprio agente define e reutiliza (`grab`, `get_meta`), inflando
# a contagem de chamadas redundantes em ~3,7×.
INV = inventario_ferramentas(RAW)
print(f"ferramentas DECLARADAS no system prompt (união de todos os papéis): {len(INV)}")

ignored = total_assign = rac_n = ram = tool_skip = 0
rac_by = Counter(); calls = Counter()
SUCCESS = re.compile(r'\b(sucesso|conclu[ií]|apliquei|foi aplicad|funcionou|pronto|obtive|com êxito)\b', re.I)
for seq in bykey.values():
    seen_traj = Counter()
    later_names = []
    trees = []
    for s in seq:
        try: trees.append(ast.parse(s["code"]))
        except Exception: trees.append(None)
    for t in trees:
        later_names.append({n.id for n in ast.walk(t) if isinstance(n, ast.Name) and isinstance(n.ctx, ast.Load)} if t else set())
    for i, (s, t) in enumerate(zip(seq, trees)):
        if s["err_type"] and SUCCESS.search(s["thought"]): ram += 1
        if t is None: continue
        fut = set().union(*later_names[i:]) if later_names[i:] else set()
        for n in ast.walk(t):
            if isinstance(n, ast.Call) and isinstance(n.func, ast.Name):
                if n.func.id in INV:
                    calls[n.func.id] += 1
                    if not s["err_type"]:
                        try: seen_traj[(n.func.id, ast.dump(n))] += 1
                        except Exception: pass
            if isinstance(n, ast.Assign) and isinstance(n.value, ast.Call) and isinstance(n.value.func, ast.Name):
                if n.value.func.id in INV and n.value.func.id != "final_answer":
                    total_assign += 1
                    tg = [x.id for x in n.targets if isinstance(x, ast.Name)]
                    if tg and not any(x in fut for x in tg): ignored += 1
    # RAC conservador: exclui final_answer — chamá-lo duas vezes pode ser comportamento do
    # harness, não redundância do agente. O número varia ~3,7× conforme essa escolha e o
    # inventário usado; ver procedimento-validacao.md §1.6.
    for key, c in seen_traj.items():
        if c > 1 and key[0] != "final_answer":
            rac_n += c-1; rac_by[key[0]] += c-1

execs_tool = defaultdict(int); execs_fin = defaultdict(bool)
for r in RAW:
    try: t = ast.parse(r["code"])
    except Exception: continue
    for n in ast.walk(t):
        if isinstance(n, ast.Call) and isinstance(n.func, ast.Name):
            if n.func.id == "final_answer": execs_fin[r["exec_id"]] = True
            elif n.func.id in INV: execs_tool[r["exec_id"]] += 1
tool_skip = sum(1 for e in execs_fin if execs_fin[e] and execs_tool.get(e,0) == 0)

print("detectores de falha silenciosa:\n")
print(f"1. Result-Ignore — retorno de ferramenta atribuído e NUNCA usado depois:")
print(f"   {ignored} de {total_assign} atribuições ({ignored/max(total_assign,1):.1%})")
print(f"\n2. RAC — chamada IDÊNTICA repetida na trajetória, em steps SEM erro: {rac_n}")
display(pd.Series(rac_by).sort_values(ascending=False).head(8).to_frame("repetições redundantes").T)
print(f"3. [RETIRADO] Reasoning-action mismatch: {ram} steps")
print(f"\n4. [PROVISÓRIO] Tool-Skip — execução chega a final_answer sem chamar ferramenta declarada:")
print(f"   {tool_skip} de {len(execs)} execuções ({tool_skip/len(execs):.1%})")
print(f"\n5. Ferramentas distintas efetivamente chamadas no código: {len(calls)}")
print(f"   (declaradas: {len(INV)})")
display(pd.Series(calls).sort_values(ascending=False).head(12).to_frame("chamadas").T)

ferramentas DECLARADAS no system prompt (união de todos os papéis): 90


detectores de falha silenciosa:

1. Result-Ignore — retorno de ferramenta atribuído e NUNCA usado depois:
   103 de 3053 atribuições (3.4%)

2. RAC — chamada IDÊNTICA repetida na trajetória, em steps SEM erro: 125


,get_available_documents,ConversationAgent,WorkflowManager,answer_question_using_documents,contact_user,resposta_final,compare_documents,get_docs_from_filters
repetições redundantes,22,17,12,12,11,8,6,5


3. [RETIRADO] Reasoning-action mismatch: 45 steps

4. [PROVISÓRIO] Tool-Skip — execução chega a final_answer sem chamar ferramenta declarada:
   10 de 840 execuções (1.2%)

5. Ferramentas distintas efetivamente chamadas no código: 80
   (declaradas: 90)


,final_answer,ConversationAgent,answer_question_using_documents,get_available_documents,WorkflowManager,trigger_worker_execution,contact_user,get_fields_definition,compare_documents,summarize_documents,busca_obf,puxa_doc_decisao
chamadas,2674,1019,821,730,167,78,76,56,47,43,41,36


## 13.8 · Tabela de ocorrências — a saída para a consolidação

Uma linha por falha silenciosa, no formato comum aos dois baldes (`exec_id`, papel, `idx`, mês, unidade, canal) mais o
que só este balde tem (ferramenta, grupo, forma). A coluna `unidade` fica vazia até a consolidação (plano 4.2b), que
traz a regra do balde invisível para o catálogo (`json_invalido` + argumento colado → `U_repr_colado`). Fica em
`resultados/evidencia/silenciosas/ocorrencias.csv` (git-ignored: tem `exec_id`).

In [9]:
import os
OC = (sil.rename(columns={"role": "papel"})
         [["exec_id", "papel", "idx", "mes", "ferramenta", "grupo", "forma"]]
         .assign(unidade=None, canal="silencioso"))
pasta = os.path.join("resultados", "evidencia", "silenciosas")
os.makedirs(pasta, exist_ok=True)
OC.to_csv(os.path.join(pasta, "ocorrencias.csv"), index=False)
print(f"{len(OC)} ocorrências silenciosas → {os.path.join(pasta, 'ocorrencias.csv')} (não sai da máquina)")
display(OC.groupby("grupo").size().to_frame("ocorrências").T)

120 ocorrências silenciosas → resultados/evidencia/silenciosas/ocorrencias.csv (não sai da máquina)


grupo,argumento_do_agente,fora_da_cobertura,json_invalido,nao_reconhecido,plataforma,sem_resultado
ocorrências,41,2,6,9,19,43
